In [1]:
from psychopy import visual, core
import numpy as np

### Ablauf
- Konstanten definieren
- LSL Stream eroeffnen
- MI Markers reinschieben


## Basics fuers Experiment 

- 50 Trials pro Klasse
- gleich viele Rechts/Links Klassen? **JA**
- - nicht mehr als 3 mal hintereinander dieselbe Klasse zeigen (um Erwartungseffekte zu verhindern)

Das Standard-Paradigma ("Graz-Paradigma" , Basis für BCI Competition IV 2a) gliedert sich in die Phasen:
| Phase | Dauer | Zweck |
|---|---|---|
| Ruhe/Fixation | 1–2 s | Baseline, Blinzeln erlaubt |
| Warn-Cue (Beep/Ton) | ~0.5–1 s | Aufmerksamkeit fokussieren, Vorbereitung |
| Richtungs-Cue (Pfeil/Text/Avatar) | erscheint danach | zeigt an, was vorgestellt werden soll |
| Motor-Imagery-Phase | 3–4 s | eigentliche Vorstellung |
| Inter-Trial-Interval (ITI) / Rest | variabel, randomisiert | Erholung, verhindert Antizipation |

In [25]:
# KONSTANTEN
RUHE_DAUER = 2
RUHE_SYMBOL = "+"

BEEP_DAUER = 1
MI_DAUER = 4
#REST soll variabel sein
MI_SYMBOL = ["RECHTS", "LINKS"]
trials = np.random.rand(10)
trials = list(map(lambda x:  0 if x < 0.5 else 1, trials))

print(trials)

[0, 0, 1, 1, 1, 0, 1, 1, 0, 0]


Wie geht es effizienter?

```np.zeros``` - legt einen zusammenhängenden Speicherblock (contiguous memory) fest: 5 Elemente × 8 Byte (int64 auf den meisten 64-bit-Systemen) = 40 Byte, initialisiert mit Nullen

NumPy-Arrays sind im Kern ein C-Array mit fixer Größe und festem Datentyp 
– anders als Python-Listen, die eigentlich Arrays von Zeigern auf Python-Objekte sind (jedes Element ein eigenes Objekt mit Overhead)

```np.concatenate```: der teuerste Schritt

concatenate kann Arrays nicht einfach "verbinden" (linken) wie z. B. verkettete Listen – NumPy-Arrays müssen contiguous im Speicher liegen. Also: NumPy allokiert einen komplett neuen, dritten Speicherblock (Größe = Summe beider Arrays) und kopiert beide Quellarrays byteweise hinein

```np.random.shuffle```: ist in-place – es alloziert keinen neuen Speicher, sondern permutiert die Elemente direkt im bestehenden Speicherblock von tr (Fisher-Yates-Algorithmus, O(n) Zeit, O(1) zusätzlicher Speicher). 

Das ist effizienter als np.random.permutation(tr), welches stattdessen eine komplette Kopie anlegt und die gemischt zurückgibt

In [42]:
# V1:
zeros = np.zeros ((5,), dtype=bool)
ones = np.ones_like(zeros, dtype=bool)
tr = np.concatenate((zeros,ones))
np.random.shuffle(tr)
print(tr)

[ True  True  True False False False  True False False  True]


In [65]:
# V2:
trials_per_class = 30
trials = np.array([0,1], dtype=np.int8)
trials = np.repeat(trials, [trials_per_class,trials_per_class]) #np.repeat übernimmt den Dtype des Eingabe-Arrays
np.random.shuffle(trials)

print("Trials list: ", trials)

Trials list:  [0 0 0 1 0 0 0 0 1 1 0 0 0 0 1 1 1 0 0 1 0 1 0 1 0 1 0 1 0 1 1 0 0 1 1 1 0
 1 0 1 1 1 1 0 1 1 1 0 0 0 0 1 1 1 1 1 0 0 0 1]


In [72]:
eeg_data = np.empty( shape=(1,4))

print(eeg_data)

# Add a row
eeg_data = np.r_ [eeg_data, [[1,2,3,4]]]
print(eeg_data)

[[1.67488254e-321 9.62024773e-312 9.63101149e-312 9.63101149e-312]]
[[1.67488254e-321 9.62024773e-312 9.63101149e-312 9.63101149e-312]
 [1.00000000e+000 2.00000000e+000 3.00000000e+000 4.00000000e+000]]


In [93]:
# Fuelle array mit random Werten zwischen 1.5 bis 2.5 in 0.25 Schritten (fuer Inter Trial Interval)
import random
print(random.uniform(0.5,2.5))

print(np.arange(1.5, 2.5, step=0.25))

# ITI Inter Trial Interval
iti = np.random.choice(np.arange(1.5, 2.5, step=0.25),
                       size=(10) )
print(iti)

2.1440686057260994
[1.5  1.75 2.   2.25]
[1.75 1.75 2.   1.5  1.5  2.25 2.25 1.75 2.   2.  ]
